# 13C Test: Current Best CNN Fine-Tune

Load the best persisted 13C commutative CNN fine-tune checkpoint from the latest CNN campaign and run evaluation only. This notebook does not train or update model weights.

In [ ]:
%load_ext autoreload
%autoreload 2


from dataclasses import asdict
from datetime import datetime
import json
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
import numpy as np
import pandas as pd
import torch

from src.dataset_config import load_current_dataset_artifact_path
from src.ml import (
    CommutativeCNNClassifier,
    LossWeightConfig,
    OptimizationConfig,
    create_experiment_run,
    display_experiment_summary,
    display_holdout_evaluation,
    prepare_multitask_experiment_data,
)
from src.models.configs import CommutativeCNNConfig
from src.training.reporting import build_confusion_matrix_frames
from src.tensor_utils import (
    build_tensor_embedding_2d,
    load_labeled_tensor_dataset,
    plot_tensor_embedding_2d,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [ ]:
# User inputs

campaign_root = Path("artifacts/campaigns/cnn_pretrain_finetune")
manual_run_dir = None  # Optional: Path("artifacts/.../outputs/13C/runs/13C_finetune_commutative_cnn_YYYYMMDD_HHMMSS")
manual_checkpoint_path = None  # Optional: Path("artifacts/.../13C_finetune_commutative_cnn_YYYYMMDD_HHMMSS_model_state.pt")
manual_config_path = None  # Optional: matching *_config.json or *_config.yaml

selection_target = "compound"
selection_metric = "macro_f1"
fallback_selection_metrics = ["weighted_f1", "accuracy", "roc_auc_ovr_macro", "average_precision_macro"]

test_output_dir = Path("artifacts/nb13C_commutative_cnn_test")
umap_method = "umap"


In [ ]:
def _read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def _latest_campaign_trial(campaign_root: Path) -> Path:
    if not campaign_root.exists():
        raise FileNotFoundError(f"Campaign root does not exist: {campaign_root}")
    state_path = campaign_root / "campaign_state.json"
    if state_path.exists():
        state = _read_json(state_path)
        trial_dir = state.get("current_trial_dir")
        if trial_dir and Path(trial_dir).exists():
            return Path(trial_dir)
    trial_dirs = sorted(
        path for path in campaign_root.iterdir()
        if path.is_dir() and path.name.startswith("cnn_pretrain_finetune_")
    )
    if not trial_dirs:
        raise FileNotFoundError(f"No CNN campaign trial directories found in {campaign_root}")
    return trial_dirs[-1]


def _candidate_config_path(run_dir: Path, experiment_id: str) -> Path | None:
    for suffix in ("json", "yaml", "yml"):
        path = run_dir / f"{experiment_id}_config.{suffix}"
        if path.exists():
            return path
    configs = sorted(run_dir.glob("*config.json")) + sorted(run_dir.glob("*config.yaml")) + sorted(run_dir.glob("*config.yml"))
    return configs[0] if configs else None


def _load_config(path: Path) -> dict:
    if path.suffix.lower() == ".json":
        return _read_json(path)
    try:
        import yaml
    except ImportError as exc:
        raise ImportError(f"PyYAML is required to read {path}") from exc
    payload = yaml.safe_load(path.read_text(encoding="utf-8"))
    return payload or {}


def _load_summary_score(summary_path: Path, target: str, metric: str, fallbacks: list[str]) -> tuple[float, str] | None:
    if not summary_path.exists():
        return None
    summary_df = pd.read_csv(summary_path)
    if not {"target", "metric", "value"}.issubset(summary_df.columns):
        return None
    target_df = summary_df[summary_df["target"].astype(str).eq(target)].copy()
    if target_df.empty:
        return None
    for metric_name in [metric, *fallbacks]:
        rows = target_df[target_df["metric"].astype(str).eq(metric_name)]
        if not rows.empty:
            return float(rows.iloc[0]["value"]), metric_name
    return None


def _discover_13c_candidates(trial_dir: Path) -> pd.DataFrame:
    rows = []
    for checkpoint_path in sorted(trial_dir.glob("outputs/13C/runs/*/*_model_state.pt")):
        run_dir = checkpoint_path.parent
        experiment_id = checkpoint_path.name.removesuffix("_model_state.pt")
        summary_path = run_dir / f"{experiment_id}_summary_metrics.csv"
        config_path = _candidate_config_path(run_dir, experiment_id)
        score = _load_summary_score(summary_path, selection_target, selection_metric, fallback_selection_metrics)
        rows.append(
            {
                "experiment_id": experiment_id,
                "run_dir": run_dir,
                "checkpoint_path": checkpoint_path,
                "config_path": config_path,
                "summary_path": summary_path if summary_path.exists() else None,
                "score": None if score is None else score[0],
                "score_metric": None if score is None else score[1],
                "checkpoint_mtime": checkpoint_path.stat().st_mtime,
            }
        )
    return pd.DataFrame(rows)


def _manual_candidate() -> pd.Series:
    if manual_checkpoint_path is None:
        raise ValueError("manual_checkpoint_path is required when using manual mode")
    checkpoint_path = Path(manual_checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Manual checkpoint does not exist: {checkpoint_path}")
    run_dir = Path(manual_run_dir) if manual_run_dir is not None else checkpoint_path.parent
    experiment_id = checkpoint_path.name.removesuffix("_model_state.pt")
    config_path = Path(manual_config_path) if manual_config_path is not None else _candidate_config_path(run_dir, experiment_id)
    if config_path is None or not config_path.exists():
        raise FileNotFoundError("Could not find the manual run config. Set manual_config_path.")
    summary_path = run_dir / f"{experiment_id}_summary_metrics.csv"
    score = _load_summary_score(summary_path, selection_target, selection_metric, fallback_selection_metrics)
    return pd.Series(
        {
            "experiment_id": experiment_id,
            "run_dir": run_dir,
            "checkpoint_path": checkpoint_path,
            "config_path": config_path,
            "summary_path": summary_path if summary_path.exists() else None,
            "score": None if score is None else score[0],
            "score_metric": None if score is None else score[1],
            "checkpoint_mtime": checkpoint_path.stat().st_mtime,
        }
    )


def _select_best_candidate(candidates: pd.DataFrame) -> pd.Series:
    if candidates.empty:
        latest_trial = _latest_campaign_trial(campaign_root)
        campaign_state_path = campaign_root / "campaign_state.json"
        state = _read_json(campaign_state_path) if campaign_state_path.exists() else {}
        stage_state_files = sorted(latest_trial.glob("stage_state/*.json"))
        stage_states = {path.stem: _read_json(path) for path in stage_state_files}
        message = (
            "No persisted 13C fine-tune checkpoint was found in the latest CNN campaign trial.\n"
            f"Latest trial: {latest_trial}\n"
            f"Campaign status: {state.get('status')} / phase={state.get('phase')} / current_stage={state.get('current_stage')}\n"
            f"Stage states: {stage_states}\n"
            "Run or finish the 13C stage first, or set manual_checkpoint_path/manual_config_path above."
        )
        raise FileNotFoundError(message)
    ranked = candidates.copy()
    ranked["has_score"] = ranked["score"].notna()
    ranked["score_for_rank"] = ranked["score"].fillna(-np.inf)
    ranked = ranked.sort_values(
        ["has_score", "score_for_rank", "checkpoint_mtime"],
        ascending=[False, False, False],
    )
    return ranked.iloc[0]


if manual_checkpoint_path is not None:
    selected = _manual_candidate()
    latest_trial = Path(manual_run_dir) if manual_run_dir is not None else selected["run_dir"].parent.parent.parent
    candidates = pd.DataFrame([selected])
else:
    latest_trial = _latest_campaign_trial(campaign_root)
    candidates = _discover_13c_candidates(latest_trial)
    selected = _select_best_candidate(candidates)

print(f"Latest CNN campaign trial: {latest_trial}")
if not candidates.empty:
    display(candidates.sort_values(["score", "checkpoint_mtime"], ascending=[False, False]))
print(f"Selected experiment: {selected['experiment_id']}")
print(f"Selected checkpoint: {Path(selected['checkpoint_path']).resolve()}")
print(f"Selected config: {Path(selected['config_path']).resolve()}")
print(f"Selection score: {selected['score']} ({selected['score_metric']})")


In [ ]:
run_config = _load_config(Path(selected["config_path"]))

model_config = CommutativeCNNConfig(**dict(run_config["model_config"]))
optimization_payload = dict(run_config["optimization_config"])
optimization_payload["device"] = None  # Use the currently available device for evaluation.
optimization_payload["verbose"] = False
optimization_config = OptimizationConfig(**optimization_payload)
loss_weight_config = LossWeightConfig(**dict(run_config["loss_weight_config"]))

dataset_artifact_path = Path(run_config["dataset_artifact_path"]) if run_config.get("dataset_artifact_path") else load_current_dataset_artifact_path()
holdout_fraction = float(run_config.get("holdout_fraction", 0.25))
validation_fraction_within_train = float(run_config.get("validation_fraction_within_train", 0.20))
train_num_random_rotations = int(run_config.get("train_num_random_rotations", 0))
rotation_range_degrees = float(run_config.get("rotation_range_degrees", 0.0))
freeze_backbone = bool(run_config.get("freeze_backbone", False))
hot_start = bool(run_config.get("hot_start", True))

experiment_run = create_experiment_run(test_output_dir, "13C_test_commutative_cnn")
figure_dir = Path(experiment_run.figure_dir)
print(f"Test-only output folder: {Path(experiment_run.run_dir).resolve()}")
print(f"Dataset artifact: {dataset_artifact_path.resolve()}")


In [ ]:
dataset = load_labeled_tensor_dataset(dataset_artifact_path)
experiment = prepare_multitask_experiment_data(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
display_experiment_summary(experiment)


In [ ]:
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
    pretrained_state_path=None,
    freeze_backbone=freeze_backbone,
    hot_start=hot_start,
)

# Rebuild fitted metadata and standardization from the saved run config without running training.
prepared = model._prepare_training_data(
    experiment.X_train,
    experiment.y_train.to_numpy(),
    validation_data=(experiment.splits.X_val, experiment.splits.y_val),
    compound_y=None if experiment.compound_train is None else experiment.compound_train.to_numpy(),
    concentration_y=None if experiment.concentration_train is None else experiment.concentration_train.to_numpy(),
    validation_compound_y=experiment.splits.compound_val,
    validation_concentration_y=experiment.splits.concentration_val,
)
model.model_ = model._build_model_from_prepared(prepared)
model.device_ = model._device()
checkpoint_payload = torch.load(Path(selected["checkpoint_path"]), map_location="cpu")
state_dict = checkpoint_payload.get("model_state_dict", checkpoint_payload) if isinstance(checkpoint_payload, dict) else checkpoint_payload
missing_keys, unexpected_keys = model.model_.load_state_dict(state_dict, strict=False)
if missing_keys or unexpected_keys:
    raise RuntimeError(
        f"Checkpoint did not match reconstructed model. missing={missing_keys}, unexpected={unexpected_keys}"
    )
model.model_.to(model.device_)
model.model_.eval()

history_path = Path(selected["run_dir"]) / f"{selected['experiment_id']}_history.csv"
if history_path.exists():
    model.history_ = pd.read_csv(history_path)

print(f"Loaded checkpoint on {model.device_}")
print(f"Action classes: {dict(zip(model.classes_, [experiment.label_maps['action'][int(k)] for k in model.classes_]))}")
if getattr(model, "compound_classes_", None) is not None:
    print(f"Compound classes: {dict(zip(model.compound_classes_, [experiment.label_maps['compound'][int(k)] for k in model.compound_classes_]))}")
if getattr(model, "concentration_classes_", None) is not None:
    print(f"Concentration classes: {dict(zip(model.concentration_classes_, [experiment.label_maps['concentration'][int(k)] for k in model.concentration_classes_]))}")


In [ ]:
holdout_evaluation = display_holdout_evaluation(model, experiment)


In [ ]:
def display_confusion_tables(name: str, y_true: dict, y_pred: dict, class_labels: dict, label_maps: dict) -> None:
    for target in y_true:
        if target not in y_pred:
            continue
        counts_df, fractions_df = build_confusion_matrix_frames(
            y_true[target],
            y_pred[target],
            class_labels=class_labels.get(target),
            label_map=label_maps.get(target),
        )
        print(f"## {name}: {target} confusion counts")
        display(counts_df)
        print(f"## {name}: {target} confusion row fractions")
        display(fractions_df)


display_confusion_tables(
    "Holdout including control",
    experiment.y_true_holdout,
    holdout_evaluation.predictions,
    experiment.class_labels,
    experiment.label_maps,
)

if holdout_evaluation.y_true_excluding_control:
    display_confusion_tables(
        "Holdout excluding control",
        holdout_evaluation.y_true_excluding_control,
        holdout_evaluation.predictions_excluding_control,
        {
            target: [label for label in labels if int(label) != 0]
            for target, labels in experiment.class_labels.items()
            if target in holdout_evaluation.y_true_excluding_control
        },
        {
            target: {label: name for label, name in label_map.items() if int(label) != 0}
            for target, label_map in experiment.label_maps.items()
            if target in holdout_evaluation.y_true_excluding_control
        },
    )


In [ ]:
holdout_embedding_projection = build_tensor_embedding_2d(
    model.transform(experiment.splits.X_holdout),
    experiment.y_true_holdout["action"],
    label_map=experiment.label_maps["action"],
    metadata=experiment.splits.metadata_holdout,
    method=umap_method,
    random_state=optimization_config.random_state,
)
holdout_embedding_projection.to_csv(
    figure_dir / f"{experiment_run.experiment_id}_holdout_embedding_umap.csv",
    index=False,
)
plot_tensor_embedding_2d(
    holdout_embedding_projection,
    title="Holdout embedding projection by action",
    marker_column="compound",
    output_path=figure_dir / f"{experiment_run.experiment_id}_holdout_embedding_umap.pdf",
)


In [ ]:
all_labeled_tensors = torch.cat(
    [
        experiment.splits.X_train_base,
        experiment.splits.X_val,
        experiment.splits.X_holdout,
    ],
    dim=0,
)
all_action_labels = torch.cat(
    [
        experiment.splits.y_train_base,
        torch.as_tensor(experiment.splits.y_val),
        torch.as_tensor(experiment.splits.y_holdout),
    ]
).numpy()
all_labeled_metadata = pd.concat(
    [
        experiment.splits.metadata_train_base.assign(dataset_split="train"),
        experiment.splits.metadata_val.assign(dataset_split="validation"),
        experiment.splits.metadata_holdout.assign(dataset_split="holdout"),
    ],
    ignore_index=True,
)

all_embedding_projection = build_tensor_embedding_2d(
    model.transform(all_labeled_tensors),
    all_action_labels,
    label_map=experiment.label_maps["action"],
    metadata=all_labeled_metadata,
    method=umap_method,
    random_state=optimization_config.random_state,
)
all_embedding_projection.to_csv(
    figure_dir / f"{experiment_run.experiment_id}_all_labeled_embedding_umap.csv",
    index=False,
)
plot_tensor_embedding_2d(
    all_embedding_projection,
    title="All labeled data embedding projection by action (with controls)",
    marker_column="compound",
    edge_color_column="dataset_split",
    edge_color_map={"train": "white", "validation": "black", "holdout": "black"},
    display_control=True,
    output_path=figure_dir / f"{experiment_run.experiment_id}_all_labeled_embedding_umap_with_controls.pdf",
)
plot_tensor_embedding_2d(
    all_embedding_projection,
    title="All labeled data embedding projection by action (controls hidden)",
    marker_column="compound",
    edge_color_column="dataset_split",
    edge_color_map={"train": "white", "validation": "black", "holdout": "black"},
    display_control=False,
    output_path=figure_dir / f"{experiment_run.experiment_id}_all_labeled_embedding_umap_controls_hidden.pdf",
)


In [ ]:
test_run_config = {
    "source_campaign_trial": latest_trial,
    "source_experiment_id": selected["experiment_id"],
    "source_run_dir": selected["run_dir"],
    "source_checkpoint_path": selected["checkpoint_path"],
    "source_config_path": selected["config_path"],
    "selection_target": selection_target,
    "selection_metric": selection_metric,
    "selection_score": selected["score"],
    "selection_score_metric": selected["score_metric"],
    "dataset_artifact_path": dataset_artifact_path,
    "test_experiment_id": experiment_run.experiment_id,
    "test_run_dir": Path(experiment_run.run_dir),
    "figure_dir": figure_dir,
    "model_config": asdict(model_config),
    "optimization_config": asdict(optimization_config),
    "loss_weight_config": asdict(loss_weight_config),
}

config_output_path = Path(experiment_run.run_dir) / f"{experiment_run.experiment_id}_config.json"
def _json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


config_output_path.write_text(
    json.dumps(_json_ready(test_run_config), indent=2, sort_keys=True),
    encoding="utf-8",
)
print(f"Saved test run config: {config_output_path.resolve()}")
print(f"Saved figures in: {figure_dir.resolve()}")
